# Étape 1 — Collecte de données (le 17 septembre 2026)

Objectif : récupérer les séries de prix des denrées alimentaires au Cameroun depuis la source https://data.humdata.org/dataset/wfp-food-prices-for-cameroon

In [24]:
# Bibliothèques

import pandas as pd
import os
import re

In [11]:
# Chargement du fichier CSV wfp_food_prices_cmr.csv

# Chemin depuis le notebook 
chemin = '../data/raw/wfp_food_prices_cmr.csv'

# Vérification
print("Chemin absolu :", os.path.abspath(chemin))
print("Existe :", os.path.exists(chemin))

if os.path.exists(chemin):
    df = pd.read_csv(chemin)
    print("✅ Chargé :", df.shape)
    display(df.head())
    display(df.tail())
else:
    print("❌ Toujours introuvable")
    print("Contenu de ../data/raw :",
          os.listdir('../data/raw') if os.path.exists('../data/raw') else "dossier absent")

Chemin absolu : /home/virlence-manga-bell/Documents/DataScience/vivres237/data/raw/wfp_food_prices_cmr.csv
Existe : True
✅ Chargé : (76807, 16)


,date,admin1,admin2,market,market_id,latitude,longitude,category,commodity,commodity_id,unit,priceflag,pricetype,currency,price,usdprice
0,2005-01-15,Centre,Mfoundi,Yaoundé-Mfoundi,2613,3.86,11.52,cereals and tubers,Maize (white),67,KG,actual,Retail,XAF,147.78,0.30
1,2005-01-15,Centre,Mfoundi,Yaoundé-Mfoundi,2613,3.86,11.52,cereals and tubers,Maize (yellow),136,KG,actual,Retail,XAF,147.78,0.30
2,2005-01-15,Littoral,Wouri,Douala-Congo,2579,4.04,9.70,cereals and tubers,Maize (white),67,KG,actual,Retail,XAF,183.64,0.37
3,2005-01-15,Littoral,Wouri,Douala-Congo,2579,4.04,9.70,cereals and tubers,Maize (yellow),136,KG,actual,Retail,XAF,183.64,0.37
4,2005-01-15,Nord,Bénoué,Garoua,1593,9.30,13.40,cereals and tubers,Maize (white),67,KG,actual,Retail,XAF,127.53,0.25


,date,admin1,admin2,market,market_id,latitude,longitude,category,commodity,commodity_id,unit,priceflag,pricetype,currency,price,usdprice
76802,2026-07-15,Sud-Ouest,Meme,Kumba,5264,4.64,9.45,cereals and tubers,"Rice (long grain, imported)",225,KG,actual,Retail,XAF,571.0,0.99
76803,2026-07-15,Sud-Ouest,Meme,Kumba,5264,4.64,9.45,cereals and tubers,Wheat flour,58,KG,actual,Retail,XAF,500.0,0.86
76804,2026-07-15,Sud-Ouest,Meme,Kumba,5264,4.64,9.45,"meat, fish and eggs","Fish (mackerel, fresh)",744,KG,actual,Retail,XAF,1996.0,3.45
76805,2026-07-15,Sud-Ouest,Meme,Kumba,5264,4.64,9.45,oil and fats,Oil (palm),62,L,actual,Retail,XAF,750.0,1.30
76806,2026-07-15,Sud-Ouest,Meme,Kumba,5264,4.64,9.45,pulses and nuts,Beans (red),78,KG,actual,Retail,XAF,550.0,0.95


# Étape 2 — Nettoyage & EDA (Exploratory Data Analysis)

<table>
  <thead>
    <tr><th>Colonne</th><th>Signification</th></tr>
  </thead>
  <tbody>
    <tr><td><code>date</code></td><td>Date de l'observation du prix (mensuelle généralement)</td></tr>
    <tr><td><code>admin1</code></td><td>Région administrative du Cameroun (ex : Centre, Littoral, Nord, Sud-Ouest)</td></tr>
    <tr><td><code>admin2</code></td><td>Département, subdivision de la région (ex : Mfoundi, Wouri, Bénoué, Meme)</td></tr>
    <tr><td><code>market</code></td><td>Marché précis où le prix a été relevé (ex : Yaoundé-Mfoundi, Douala-Congo, Garoua, Kumba)</td></tr>
    <tr><td><code>market_id</code></td><td>Identifiant numérique unique du marché (référence interne WFP)</td></tr>
    <tr><td><code>latitude</code> / <code>longitude</code></td><td>Coordonnées GPS du marché</td></tr>
    <tr><td><code>category</code></td><td>Catégorie du produit (ex : cereals and tubers, meat, fish and eggs, oil and fats, pulses and nuts)</td></tr>
    <tr><td><code>commodity</code></td><td>Produit précis (ex : Maize (white), Rice (long grain, imported), Oil (palm), Beans (red))</td></tr>
    <tr><td><code>commodity_id</code></td><td>Identifiant numérique unique du produit</td></tr>
    <tr><td><code>unit</code></td><td>Unité de mesure du prix (KG, L)</td></tr>
    <tr><td><code>priceflag</code></td><td>Type de donnée : <code>actual</code> = prix réellement observé</td></tr>
    <tr><td><code>pricetype</code></td><td><code>Retail</code> (détail) ou <code>Wholesale</code> (gros)</td></tr>
    <tr><td><code>currency</code></td><td>Devise, ici XAF (franc CFA)</td></tr>
    <tr><td><code>price</code></td><td>Prix dans la devise locale (XAF) pour l'unité donnée</td></tr>
    <tr><td><code>usdprice</code></td><td>Le même prix converti en dollars US (au taux de change du moment)</td></tr>
  </tbody>
</table>

In [ ]:
# Vu d'ensemble du DataFrame

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 76807 entries, 0 to 76806
Data columns (total 16 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   date          76807 non-null  str    
 1   admin1        76807 non-null  str    
 2   admin2        76807 non-null  str    
 3   market        76807 non-null  str    
 4   market_id     76807 non-null  int64  
 5   latitude      76807 non-null  float64
 6   longitude     76807 non-null  float64
 7   category      76807 non-null  str    
 8   commodity     76807 non-null  str    
 9   commodity_id  76807 non-null  int64  
 10  unit          76807 non-null  str    
 11  priceflag     76807 non-null  str    
 12  pricetype     76807 non-null  str    
 13  currency      76807 non-null  str    
 14  price         76807 non-null  float64
 15  usdprice      76807 non-null  float64
dtypes: float64(4), int64(2), str(10)
memory usage: 9.4 MB


## Nettoyage

In [10]:
# Valeurs manquantes

# Compter les NaN par colonne

# Nombre de valeurs manquantes
df.isna().sum()

# Chaînes vides
(df == '').sum()

# Valeurs textuelles signifiant "manquant"
for v in ['NA', 'N/A', 'null', 'None', '-', '?', 'unknown', '']:
    n = (df.astype(str).apply(lambda c: c.str.strip().str.lower() == v.lower())).sum().sum()
    if n:
        print(f"{v!r:>10} → {n} occurrences")

# En pourcentage
(df.isna().sum() / len(df) * 100).round(2).sort_values(ascending=False)

date            0.0
admin1          0.0
admin2          0.0
market          0.0
market_id       0.0
latitude        0.0
longitude       0.0
category        0.0
commodity       0.0
commodity_id    0.0
unit            0.0
priceflag       0.0
pricetype       0.0
currency        0.0
price           0.0
usdprice        0.0
dtype: float64

In [13]:
# Comprendre priceflag et pricetype (évite le piège des données projetées)

print("priceflag :")
print(df['priceflag'].value_counts())
print("\npricetype :")
print(df['pricetype'].value_counts())
print("\ncurrency :")
print(df['currency'].value_counts())

priceflag :
priceflag
actual    76807
Name: count, dtype: int64

pricetype :
pricetype
Retail       57512
Wholesale    19295
Name: count, dtype: int64

currency :
currency
XAF    76807
Name: count, dtype: int64


In [14]:
# Étendue temporelle et cohérence des dates

df['date'] = pd.to_datetime(df['date'])
print("Première date :", df['date'].min())
print("Dernière date :", df['date'].max())
print("Nombre de dates uniques :", df['date'].nunique())

# Fréquence : mensuelle ? irrégulière ?
print(df['date'].dt.to_period('M').value_counts().sort_index().head(15))

Première date : 2005-01-15 00:00:00
Dernière date : 2026-07-15 00:00:00
Nombre de dates uniques : 259
date
2005-01    10
2005-02    10
2005-03    10
2005-04    10
2005-05    10
2005-06    10
2005-07    10
2005-08    10
2005-09    10
2005-10    10
2005-11    10
2005-12    10
2006-01    10
2006-02    10
2006-03    10
Freq: M, Name: count, dtype: int64


In [15]:
# Doublons — un même (date, marché, produit) apparaît-il plusieurs fois ?

doublons = df.duplicated(subset=['date', 'market_id', 'commodity_id', 'pricetype'], keep=False)
print("Nombre de lignes dupliquées :", doublons.sum())
df[doublons].sort_values(['market_id', 'commodity_id', 'date']).head(10)

Nombre de lignes dupliquées : 0


,date,admin1,admin2,market,market_id,latitude,longitude,category,commodity,commodity_id,unit,priceflag,pricetype,currency,price,usdprice


In [16]:
# Couverture géographique et produits

print("Régions (admin1) :", df['admin1'].nunique(), "→", sorted(df['admin1'].unique()))
print("\nDépartements (admin2) :", df['admin2'].nunique())
print("\nMarchés :", df['market'].nunique())
print("\nCatégories :", df['category'].nunique(), "→", sorted(df['category'].unique()))
print("\nProduits (commodity) :", df['commodity'].nunique())
df['commodity'].value_counts()

Régions (admin1) : 9 → ['Adamaoua', 'Centre', 'Est', 'Extrême-Nord', 'Littoral', 'Nord', 'Nord-Ouest', 'Ouest', 'Sud-Ouest']

Départements (admin2) : 30

Marchés : 83

Catégories : 8 → ['cereals and tubers', 'meat, fish and eggs', 'milk and dairy', 'miscellaneous food', 'non-food', 'oil and fats', 'pulses and nuts', 'vegetables and fruits']

Produits (commodity) : 55


commodity
Maize (white)                    4778
Groundnuts (shelled)             4277
Maize (yellow)                   3999
Rice (long grain, imported)      3755
Beans (niebe)                    3523
Onions                           3402
Meat (beef)                      3043
Oil (palm)                       3030
Sugar                            2404
Salt                             2313
Sorghum (red)                    2266
Fish (mackerel, fresh)           2137
Sorghum (white)                  2124
Eggs                             2113
Beans (red)                      2064
Rice (local)                     2027
Plantains                        1892
Cassava (fresh)                  1866
Cocoyam (macabo)                 1854
Potatoes                         1783
Wheat flour                      1595
Milk (powder)                    1563
Cassava (cossette)               1554
Oil (cotton)                     1300
Bananas                          1259
Potatoes (Irish)                 1221
To

In [17]:
# Valeurs de prix aberrantes

print(df[['price', 'usdprice']].describe())

# Prix nuls ou négatifs (ne devraient pas exister)
print("\nPrix <= 0 :", (df['price'] <= 0).sum())
print("Usdprice <= 0 :", (df['usdprice'] <= 0).sum())

# Cohérence price/usdprice : taux de change implicite, doit être stable dans le temps
df['taux_implicite'] = df['price'] / df['usdprice']
print("\nTaux de change implicite (XAF/USD) :")
print(df['taux_implicite'].describe())

               price      usdprice
count   76807.000000  76807.000000
mean     5430.176968      9.534128
std     10944.594885     19.460332
min         3.000000      0.005000
25%       410.000000      0.700000
50%       900.000000      1.490000
75%      3011.750000      5.160000
max    160000.000000    262.300000

Prix <= 0 : 0
Usdprice <= 0 : 0

Taux de change implicite (XAF/USD) :
count    76807.000000
mean       592.989637
std         99.741159
min        408.547170
25%        571.808511
50%        598.924731
75%        614.234772
max       6200.000000
Name: taux_implicite, dtype: float64


In [19]:
# Le taux de change implicite (XAF/USD) varie de 408 à 6200, avec une moyenne à 593 et un écart-type de ~100. Le taux XAF/USD réel a historiquement oscillé entre ~500 et ~650 sur la période 2005-2026. Une valeur max de 6200 est clairement une anomalie — soit une erreur de saisie sur price ou usdprice pour certaines lignes, soit un problème d'unité.

# Isoler les taux de change suspects
anomalies = df[(df['taux_implicite'] < 450) | (df['taux_implicite'] > 750)]
print("Nombre de lignes suspectes :", len(anomalies))
anomalies[['date', 'market', 'commodity', 'price', 'usdprice', 'taux_implicite']].sort_values('taux_implicite', ascending=False).head(15)

Nombre de lignes suspectes : 148


,date,market,commodity,price,usdprice,taux_implicite
42873,2023-03-15,Kumba,Maize (yellow),6200.0,1.0,6200.0
57546,2024-04-15,Nkambe,Cassava (fresh),6150.0,1.0,6150.0
56870,2024-03-15,Douala-Marché Central,Maize (yellow),6025.0,1.0,6025.0
6902,2015-12-15,Douala-Marché Central,Cocoyam (macabo),6000.0,1.0,6000.0
9268,2017-05-15,Mada,Maize (white),18000.0,3.0,6000.0
6912,2015-12-15,Douala-Sandaga,Cocoyam (macabo),6000.0,1.0,6000.0
6869,2015-12-15,Maroua,Sorghum (red),12000.0,2.0,6000.0
9243,2017-05-15,Logone-Birni,Maize (white),18000.0,3.0,6000.0
6890,2015-12-15,Douala-Bonaberi,Cocoyam (macabo),6000.0,1.0,6000.0
9269,2017-05-15,Mada,Maize (yellow),18000.0,3.0,6000.0


In [20]:
# Les lignes suspectes utilisent-elles une unité différente ?

print(df['unit'].value_counts())
print("\nUnités des lignes suspectes uniquement :")
print(anomalies['unit'].value_counts())

unit
KG         48054
90 KG       7341
L           5411
100 KG      3749
1 piece     2113
400 G       1563
5 KG        1392
12 KG       1392
18 KG       1320
20 KG       1237
50 KG        978
20 L         623
160 KG       547
15 KG        540
Day          371
120 KG       176
Name: count, dtype: int64

Unités des lignes suspectes uniquement :
unit
KG        122
90 KG      15
20 KG       3
100 KG      2
20 L        2
50 KG       1
18 KG       1
15 KG       1
160 KG      1
Name: count, dtype: int64


In [21]:
# Ces prix sont-ils cohérents avec les prix du même produit, même marché, à d'autres dates proches ?

# Exemple : maïs blanc à Mada, comparé aux dates autour de 2017-05-15
mada_maize = df[(df['market'] == 'Mada') & (df['commodity'] == 'Maize (white)')].sort_values('date')
mada_maize[['date', 'price', 'usdprice', 'unit']]

,date,price,usdprice,unit
729,2011-01-15,16000.0,32.55,90 KG
774,2011-02-15,18000.0,37.01,90 KG
820,2011-03-15,18800.0,40.12,90 KG
865,2011-04-15,21000.0,46.21,90 KG
911,2011-05-15,20600.0,44.42,90 KG
...,...,...,...,...
14429,2020-02-15,17250.0,28.24,90 KG
18606,2021-03-15,567.0,1.04,KG
19282,2021-04-15,350.0,0.64,KG
20184,2021-06-15,300.0,0.56,KG


In [25]:
# Normaliser les prix par unité

# Créons une colonne price_per_kg (ou price_per_liter/price_per_unit selon le cas) qui ramène tout à une base comparable :

def extraire_multiplicateur(unite):
    """Retourne (quantité, unité_de_base) à partir d'une chaîne comme '90 KG', 'L', '400 G'."""
    unite = unite.strip()
    match = re.match(r'^(\d+)\s*(KG|G|L)$', unite)
    if match:
        qte, base = match.groups()
        qte = float(qte)
        if base == 'G':
            qte = qte / 1000  # conversion en kg
            base = 'KG'
        return qte, base
    elif unite == 'KG':
        return 1.0, 'KG'
    elif unite == 'L':
        return 1.0, 'L'
    else:
        return None, unite  # unités non-poids/volume : "1 piece", "Day"

df[['quantite', 'unite_base']] = df['unit'].apply(lambda u: pd.Series(extraire_multiplicateur(u)))

# Prix normalisé (par kg ou par litre selon le produit)
df['price_per_unit'] = df['price'] / df['quantite']

# Vérifier les unités qu'on n'a pas su convertir
print("Unités non converties :")
print(df[df['quantite'].isna()]['unit'].value_counts())

Unités non converties :
unit
1 piece    2113
Day         371
Name: count, dtype: int64


In [26]:
# Refaire la comparaison à la médiane annuelle mais cette fois sur price_per_unit, ce qui devrait faire disparaître la grande majorité des "anomalies" :

df['annee'] = df['date'].dt.year
mediane_normalisee = df.groupby(['annee', 'commodity', 'unite_base'])['price_per_unit'].median()

df_verif = df.dropna(subset=['price_per_unit']).copy()
df_verif['prix_median_norm'] = df_verif.apply(
    lambda r: mediane_normalisee.get((r['annee'], r['commodity'], r['unite_base']), None), axis=1
)
df_verif['ratio_norm'] = df_verif['price_per_unit'] / df_verif['prix_median_norm']

vrais_suspects = df_verif[(df_verif['ratio_norm'] < 0.3) | (df_verif['ratio_norm'] > 3)]
print("Vrais suspects après normalisation :", len(vrais_suspects))
vrais_suspects[['date', 'market', 'commodity', 'unit', 'price', 'price_per_unit', 'prix_median_norm', 'ratio_norm']].sort_values('ratio_norm', ascending=False)

Vrais suspects après normalisation : 1259


,date,market,commodity,unit,price,price_per_unit,prix_median_norm,ratio_norm
57141,2024-04-15,Yaoundé-Mfoundi,"Rice (long grain, imported)",KG,47000.0,47000.0,600.00,78.333333
57155,2024-04-15,Yaoundé-Mokolo,"Rice (long grain, imported)",KG,30200.0,30200.0,600.00,50.333333
57329,2024-04-15,Yagoua,Wheat flour,KG,29000.0,29000.0,581.50,49.871023
57142,2024-04-15,Yaoundé-Mfoundi,Wheat flour,KG,28400.0,28400.0,581.50,48.839209
57452,2024-04-15,Douala-Marché Central,Wheat flour,KG,26125.0,26125.0,581.50,44.926913
...,...,...,...,...,...,...,...,...
39625,2023-01-15,Guider,Okra (dry),KG,6.0,6.0,905.00,0.006630
39166,2023-01-15,Gazawa,Tomatoes,KG,3.0,3.0,500.00,0.006000
39373,2023-01-15,Yagoua,Tomatoes,KG,3.0,3.0,500.00,0.006000
39164,2023-01-15,Gazawa,Okra (fresh),KG,3.0,3.0,848.00,0.003538


In [27]:
# Nombre de lignes avant suppression
print("Avant suppression :", len(df))

# On supprime les lignes identifiées comme vraies anomalies (par leur index)
df_clean = df.drop(index=vrais_suspects.index).copy()

print("Après suppression :", len(df_clean))
print("Lignes supprimées :", len(df) - len(df_clean))
print("Pourcentage supprimé :", round((len(df) - len(df_clean)) / len(df) * 100, 2), "%")

Avant suppression : 76807
Après suppression : 75548
Lignes supprimées : 1259
Pourcentage supprimé : 1.64 %


In [28]:
# Vérifier qu'aucun produit ou marché n'a été vidé accidentellement
print("Produits uniques avant :", df['commodity'].nunique())
print("Produits uniques après :", df_clean['commodity'].nunique())
print("\nMarchés uniques avant :", df['market'].nunique())
print("Marchés uniques après :", df_clean['market'].nunique())

Produits uniques avant : 55
Produits uniques après : 55

Marchés uniques avant : 83
Marchés uniques après : 83


In [22]:
# Ces anomalies sont-elles concentrées sur certaines dates précises (erreur de saisie ponctuelle lors d'une collecte) ?

print(anomalies['date'].value_counts().sort_index())

date
2007-11-15     8
2008-01-15    10
2008-02-15     8
2008-03-15    10
2008-04-15    10
2008-05-15    10
2008-06-15    10
2008-07-15    10
2008-08-15    10
2009-09-15    10
2009-10-15    10
2009-11-15    10
2009-12-15     4
2011-04-15     2
2012-04-15     5
2014-07-15     3
2015-12-15     4
2017-05-15     5
2018-06-15     3
2021-06-15     3
2023-03-15     1
2024-03-15     1
2024-04-15     1
Name: count, dtype: int64


In [23]:
# Comparaison au prix médian du même produit, la même année, tous marchés confondus

df['annee'] = df['date'].dt.year
mediane_annuelle = df.groupby(['annee', 'commodity'])['price'].median()

# Rejoindre pour comparer chaque ligne suspecte à la médiane de son année/produit
anomalies_check = anomalies.copy()
anomalies_check['annee'] = anomalies_check['date'].dt.year
anomalies_check['prix_median_annee'] = anomalies_check.apply(
    lambda r: mediane_annuelle.get((r['annee'], r['commodity']), None), axis=1
)
anomalies_check['ratio_vs_median'] = anomalies_check['price'] / anomalies_check['prix_median_annee']
anomalies_check[['date', 'market', 'commodity', 'price', 'prix_median_annee', 'ratio_vs_median']].sort_values('ratio_vs_median', ascending=False)

,date,market,commodity,price,prix_median_annee,ratio_vs_median
1367,2012-04-15,Logone-Birni,Maize (white),20000.00,345.250,57.929037
9269,2017-05-15,Mada,Maize (yellow),18000.00,345.905,52.037409
9244,2017-05-15,Logone-Birni,Maize (yellow),18000.00,345.905,52.037409
56870,2024-03-15,Douala-Marché Central,Maize (yellow),6025.00,449.500,13.403782
42873,2023-03-15,Kumba,Maize (yellow),6200.00,490.000,12.653061
...,...,...,...,...,...,...
367,2008-01-15,Bamenda,Maize (yellow),137.33,197.565,0.695113
366,2008-01-15,Bamenda,Maize (white),137.33,197.565,0.695113
1387,2012-04-15,Garoua,Onions,10000.00,15000.000,0.666667
899,2011-04-15,Bamenda,Maize (white),215.89,347.630,0.621034


In [18]:
# Un produit a-t-il une couverture homogène dans le temps et l'espace ?

# Exemple pour le maïs blanc : combien de marchés le rapportent, sur quelle période
maize = df[df['commodity'] == 'Maize (white)']
print("Marchés couvrant le maïs blanc :", maize['market'].nunique())
print("Période :", maize['date'].min(), "→", maize['date'].max())
maize.groupby(maize['date'].dt.year)['market'].nunique()

Marchés couvrant le maïs blanc : 79
Période : 2005-01-15 00:00:00 → 2026-07-15 00:00:00


date
2005     5
2006     5
2007     5
2008     5
2009     5
2010     5
2011     9
2012     8
2013    19
2014    19
2015    25
2016    24
2017    13
2018    24
2019    31
2020    43
2021    45
2022    48
2023    51
2024    47
2025    25
2026    28
Name: market, dtype: int64

In [30]:
# Avant filtrage
print("Avant :", len(df_clean))

# On garde uniquement les lignes normalisables en KG ou L
df_final = df_clean[df_clean['unite_base'].isin(['KG', 'L'])].copy()

print("Après :", len(df_final))
print("Lignes exclues :", len(df_clean) - len(df_final))

# Vérification rapide
print("\nProduits uniques :", df_final['commodity'].nunique())
print("Marchés uniques :", df_final['market'].nunique())
print("\nRépartition unite_base :")
print(df_final['unite_base'].value_counts())

Avant : 75548
Après : 73064
Lignes exclues : 2484

Produits uniques : 52
Marchés uniques : 83

Répartition unite_base :
unite_base
KG    67053
L      6011
Name: count, dtype: int64


In [32]:
# Sauvegarde du DataFrame nettoyé dans un nouveau fichier CSV

df_final.to_csv('../data/processed/prix_alimentaires_nettoyes.csv', index=False)
print("Sauvegardé :", df_final.shape)

Sauvegardé : (73064, 21)
